# 1. Mapillary Image Download with Quad-Tree Division

**What it does:** Downloads all Mapillary street-level images within a defined bounding box (BBOX)

**Key Features:**
- **Quad-Tree Algorithm**: Solves Mapillary's 1000-image API limit by recursively dividing the area into smaller tiles
- **Automatic Retry**: Handles 500/timeout errors with exponential backoff
- **Multi-Format Support**: JPEG, WebP, PNG image formats
- **Metadata Export**: Creates CSV with image_id, captured_at, username, lat, lon, filename

**Configuration:**
- Update `ACCESS_TOKEN` with your Mapillary API key
- Set `ROOT_BBOX` to your desired area (West, South, East, North)

**Output:**
- Images saved to `data/mapillary/` folder
- `metadata.csv` with all geographic and temporal metadata

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
All Mapillary tiles within BBOX (limit-1000 issue solved):
- Collect id list using quad-tree division
- For each id, retrieve captured_at, geometry, thumb_1024_url, creator.username
- Download images to data/mapillary folder, write metadata.csv (lat,lon)
"""
import os, csv, time, requests, sys
from pathlib import Path
from datetime import datetime, timezone
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ACCESS_TOKEN = "MLY|9723292547755211|b7b5acc3dfcc964087ea8cfa76867fc5"
ROOT_BBOX = (28.9680, 41.0200, 28.9900, 41.0280)   # (W,S,E,N)
LIST_LIMIT = 1000
SLEEP      = 0.20
FIELDS_ITEM = "thumb_1024_url,captured_at,geometry,creator"
MAX_DEPTH  = 6
MIN_SIZE   = 1e-4   # ≈11 m; don't split into smaller tiles

# Output directory (relative to current working directory)
OUT_DIR = Path("data") / "mapillary"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- SESSION + RETRY ----------
SESSION = requests.Session()
SESSION.mount(
    "https://",
    HTTPAdapter(
        max_retries=Retry(
            total=5, backoff_factor=0.5,
            status_forcelist=[502, 503, 504],
            raise_on_status=False
        )
    )
)
READ_TIMEOUT = (10, 90)
# ------------------------------------

def ts_ms_to_iso(ms):
    return datetime.fromtimestamp(ms/1000, tz=timezone.utc).isoformat()

def detect_ext(url):
    base = url.split("?")[0]
    ext  = os.path.splitext(base)[1].lower()
    if ext:
        return ext
    try:
        ctype = SESSION.head(url, timeout=(10, 20)).headers.get("Content-Type","")
    except requests.RequestException:
        ctype = ""
    return {"image/jpeg":".jpg","image/webp":".webp","image/png":".png"}.get(ctype,".jpg")

def get_json(url, retries=5):
    """Retry on 500/timeout; otherwise return empty data."""
    for k in range(retries):
        try:
            r = SESSION.get(url, timeout=READ_TIMEOUT)
            if r.status_code >= 500:
                time.sleep(1.5 * (k+1))
                continue
            r.raise_for_status()
            return r.json()
        except (requests.exceptions.HTTPError,
                requests.exceptions.ReadTimeout,
                requests.exceptions.ConnectionError):
            time.sleep(1.5 * (k+1))
    return {"data": []}

def fetch_ids(bbox, depth=0):
    w,s,e,n = bbox
    if (e-w) < MIN_SIZE or (n-s) < MIN_SIZE:
        return set()

    url = (
        "https://graph.mapillary.com/images"
        f"?access_token={ACCESS_TOKEN}"
        f"&bbox={w},{s},{e},{n}"
        f"&fields=id"
        f"&limit={LIST_LIMIT}"
    )
    data = get_json(url)
    ids  = [d["id"] for d in data.get("data", [])]

    if len(ids) < LIST_LIMIT or depth >= MAX_DEPTH:
        return set(ids)

    mid_x, mid_y = (w+e)/2, (s+n)/2
    quads = [
        (w,     s,     mid_x, mid_y),  # SW
        (mid_x, s,     e,     mid_y),  # SE
        (w,     mid_y, mid_x, n),      # NW
        (mid_x, mid_y, e,     n),      # NE
    ]
    result = set()
    for qb in quads:
        result.update(fetch_ids(qb, depth+1))
    return result

# ---------- MAIN FLOW ----------
print("📥  Collecting image_id list (quad-tree)…")
ids = fetch_ids(ROOT_BBOX)
print(f"→  {len(ids):,} ids found.")
if not ids:
    sys.exit("⚠️  No tiles found in this region; expand the BBOX.")

metadata = []
print("📸  Downloading images…")
for img_id in tqdm(ids):
    item = get_json(
        f"https://graph.mapillary.com/{img_id}"
        f"?access_token={ACCESS_TOKEN}&fields={FIELDS_ITEM}"
    )
    if not item:
        continue
    thumb = item.get("thumb_1024_url")
    if not thumb or "geometry" not in item:
        continue

    lon, lat = item["geometry"]["coordinates"]
    ext      = detect_ext(thumb)
    fname    = f"{img_id}{ext}"
    fpath    = OUT_DIR / fname
    if not fpath.exists():
        try:
            with SESSION.get(thumb, stream=True, timeout=READ_TIMEOUT) as r:
                r.raise_for_status()
                with open(fpath, "wb") as f:
                    for chunk in r.iter_content(8192):
                        f.write(chunk)
        except Exception:
            continue

    metadata.append({
        "image_id":    img_id,
        "captured_at": ts_ms_to_iso(item["captured_at"]) if item.get("captured_at") else "",
        "username":    item.get("creator", {}).get("username",""),
        "lat":         lat,
        "lon":         lon,
        "filename":    fname
    })
    time.sleep(SLEEP)

csv_path = OUT_DIR / "metadata.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image_id","captured_at","username","lat","lon","filename"])
    writer.writeheader(); writer.writerows(metadata)

print(f"\n✅  {len(metadata):,} images downloaded • {csv_path}")

# 2. Filter Images by Polygon (Karaköy Area)

**What it does:** Filters downloaded images to only those within a specific polygon boundary

**Process:**
1. Loads `metadata.csv` from previous step
2. Creates GeoDataFrame with point geometries from lat/lon coordinates
3. Filters points inside the Karaköy polygon (Istanbul, Turkey)
4. Copies filtered images to `data/mapillary/karakoy/` subfolder

**Configuration:**
- Update `polygon_coords` to define your area of interest

**Output:** Filtered images in `karakoy/` subfolder

**Use Case:** Extract images from a specific neighborhood or area of interest within the larger BBOX

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
from pathlib import Path
import shutil

# ------------------------------------
# 1. File path and polygon definition
# ------------------------------------
MAP_DIR = Path("data") / "mapillary"
csv_path = MAP_DIR / "metadata.csv"
df = pd.read_csv(csv_path)

polygon_coords = [
    (28.97201188246599, 41.027961514127306),
    (28.972796376763487, 41.02567058090895),
    (28.97595966022113, 41.02547966620972),
    (28.982589902348348, 41.028171512351044),
    (28.986841355315423, 41.02924058383375),
    (28.98772707468357, 41.02939330691432),
    (28.98954912595517, 41.02809514943808),
    (28.99051076412629, 41.02637696048352),
    (28.98522175418511, 41.02311227795534),
    (28.9803376445265, 41.02127940279426),
    (28.97727558613951, 41.02040113203624),
    (28.976187416630072, 41.02074480459719),
    (28.975200472191293, 41.02124121735267),
    (28.973783321202273, 41.02143214433928),
    (28.971784126057038, 41.02156579290059),
    (28.96986084971479, 41.02210038443376),
    (28.968443698725757, 41.02315046231212),
    (28.967684510695936, 41.02404778832296),
    (28.967279610413353, 41.02460145146428),
    (28.967380835484, 41.02536511712449),
    (28.968038798443192, 41.02584240366503),
    (28.96869676140238, 41.02668241957435),
    (28.96889921154367, 41.02694969511649),
    (28.97201188246599, 41.027961514127306)
]
polygon = Polygon(polygon_coords)

# ------------------------------------
# 2. Create GeoDataFrame and filter
# ------------------------------------
gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.lon, df.lat)],
    crs="EPSG:4326"
)
gdf_filtered = gdf[gdf.geometry.within(polygon)]

# ------------------------------------
# 3. Create target folder
# ------------------------------------
karakoy_dir = MAP_DIR / "karakoy"
karakoy_dir.mkdir(exist_ok=True)

# ------------------------------------
# 4. Copy files
# ------------------------------------
copied = 0
for filename in gdf_filtered["filename"]:
    src = MAP_DIR / filename
    dst = karakoy_dir / filename
    if src.exists():
        shutil.copy2(src, dst)
        copied += 1

print(f"{copied} images copied to '{karakoy_dir}' folder.")

# 3. Visualize with OpenStreetMap Overlay

**What it does:** Creates a map visualization combining Mapillary points with OpenStreetMap building and road data

**Features:**
- **OSM Building Layer**: Gray building footprints as background
- **OSM Road Network**: Street grid overlay
- **Mapillary Points**: All image locations within the polygon shown as red points
- **High-Quality Output**: Saves as PDF (300 DPI) for publication

**Data Source:** Uses OSMnx to fetch real building and road geometries from OpenStreetMap

**Output:** `mapillary_osm_plot.pdf` in current directory

**Visual Style:** Clean, minimal design suitable for reports and presentations

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import osmnx as ox
from pathlib import Path

# ------------------------------------
# 1. Load data from CSV
# ------------------------------------
csv_path = Path("data") / "mapillary" / "metadata.csv"
df = pd.read_csv(csv_path)

# ------------------------------------
# 2. Convert to GeoDataFrame
# ------------------------------------
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.lon, df.lat),
    crs="EPSG:4326"
)

# ------------------------------------
# 3. Polygon (won't be visible but used for filtering)
# ------------------------------------
polygon_coords = [
    (28.97201188246599, 41.027961514127306),
    (28.972796376763487, 41.02567058090895),
    (28.97595966022113, 41.02547966620972),
    (28.982589902348348, 41.028171512351044),
    (28.986841355315423, 41.02924058383375),
    (28.98772707468357, 41.02939330691432),
    (28.98954912595517, 41.02809514943808),
    (28.99051076412629, 41.02637696048352),
    (28.98522175418511, 41.02311227795534),
    (28.9803376445265, 41.02127940279426),
    (28.97727558613951, 41.02040113203624),
    (28.976187416630072, 41.02074480459719),
    (28.975200472191293, 41.02124121735267),
    (28.973783321202273, 41.02143214433928),
    (28.971784126057038, 41.02156579290059),
    (28.96986084971479, 41.02210038443376),
    (28.968443698725757, 41.02315046231212),
    (28.967684510695936, 41.02404778832296),
    (28.967279610413353, 41.02460145146428),
    (28.967380835484, 41.02536511712449),
    (28.968038798443192, 41.02584240366503),
    (28.96869676140238, 41.02668241957435),
    (28.96889921154367, 41.02694969511649),
    (28.97201188246599, 41.027961514127306)
]
polygon = Polygon(polygon_coords)

# Filter points (only inside polygon)
gdf = gdf[gdf.within(polygon)]

# ------------------------------------
# 4. OSM Data (Inside polygon)
# ------------------------------------
tags_bld = {"building": True}
tags_road = {"highway": True}

buildings = ox.features_from_polygon(polygon, tags_bld)
roads = ox.features_from_polygon(polygon, tags_road)

# ------------------------------------
# 5. Plot (Polygon not drawn!)
# ------------------------------------
fig, ax = plt.subplots(figsize=(16, 16))

# Buildings
if not buildings.empty:
    buildings.plot(ax=ax, facecolor="#eeeeee", edgecolor="gray", linewidth=0.3)

# Roads
if not roads.empty:
    roads.plot(ax=ax, color="dimgray", linewidth=0.6)

# Points (single color)
gdf.plot(ax=ax, color="crimson", markersize=4, alpha=0.8, label="Mapillary Points")

# Settings
ax.set_title("Mapillary + OSM ", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_aspect("equal")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

# Save as PDF
output_path = Path("mapillary_osm_plot.pdf")
fig.savefig(output_path, format="pdf", dpi=300)

# 4. Temporal Analysis - Yearly Coverage Maps

**What it does:** Generates separate map visualizations for each year of data collection

**Process:**
1. Extracts year from `captured_at` timestamp (handles mixed ISO formats)
2. Groups all Mapillary points by year
3. Creates individual OSM overlay maps for each year
4. Shows temporal evolution of street-level coverage

**Output:** Interactive display of yearly maps showing data collection progression

**Use Case:** Track how street-level imagery coverage expanded over time, identify data gaps by year

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import osmnx as ox
from pathlib import Path

# ------------------------------------
# 1. File and polygon settings
# ------------------------------------
csv_path = Path("data") / "mapillary" / "metadata.csv"
df = pd.read_csv(csv_path)

polygon_coords = [
    (28.97201188246599, 41.027961514127306),
    (28.972796376763487, 41.02567058090895),
    (28.97595966022113, 41.02547966620972),
    (28.982589902348348, 41.028171512351044),
    (28.986841355315423, 41.02924058383375),
    (28.98772707468357, 41.02939330691432),
    (28.98954912595517, 41.02809514943808),
    (28.99051076412629, 41.02637696048352),
    (28.98522175418511, 41.02311227795534),
    (28.9803376445265, 41.02127940279426),
    (28.97727558613951, 41.02040113203624),
    (28.976187416630072, 41.02074480459719),
    (28.975200472191293, 41.02124121735267),
    (28.973783321202273, 41.02143214433928),
    (28.971784126057038, 41.02156579290059),
    (28.96986084971479, 41.02210038443376),
    (28.968443698725757, 41.02315046231212),
    (28.967684510695936, 41.02404778832296),
    (28.967279610413353, 41.02460145146428),
    (28.967380835484, 41.02536511712449),
    (28.968038798443192, 41.02584240366503),
    (28.96869676140238, 41.02668241957435),
    (28.96889921154367, 41.02694969511649),
    (28.97201188246599, 41.027961514127306)
]
polygon = Polygon(polygon_coords)


# ------------------------------------
# 2. Create year column (tolerant to all ISO formats)
# ------------------------------------
df["year"] = pd.to_datetime(df["captured_at"], format="mixed").dt.year


# ------------------------------------
# 3. Create GeoDataFrame
# ------------------------------------
df["geometry"] = [Point(xy) for xy in zip(df.lon, df.lat)]
gdf_all = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

# ------------------------------------
# 4. Separate map for each year
# ------------------------------------
unique_years = sorted(gdf_all["year"].dropna().unique())

for year in unique_years:
    gdf = gdf_all[(gdf_all["year"] == year) & (gdf_all.geometry.within(polygon))]

    if gdf.empty:
        continue  # skip if no data for that year

    print(f"Year: {year}, point count: {len(gdf)}")

    # OSM data fetched only for this year's points (within general polygon)
    tags_bld = {"building": True}
    tags_road = {"highway": True}

    buildings = ox.features_from_polygon(polygon, tags_bld)
    roads = ox.features_from_polygon(polygon, tags_road)

    # Map plotting
    fig, ax = plt.subplots(figsize=(18, 20))

    if not buildings.empty:
        buildings.plot(ax=ax, facecolor="#eeeeee", edgecolor="gray", linewidth=0.3)

    if not roads.empty:
        roads.plot(ax=ax, color="dimgray", linewidth=0.6)

    gdf.plot(ax=ax, color="crimson", markersize=40, alpha=0.8, label=f"{year} Mapillary")

    ax.set_title(f"Mapillary + OSM - {year}", fontsize=14)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.set_aspect("equal")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

    # Save each year as separate file (optional)
    # fig.savefig(f"mapillary_{year}.png", dpi=300)

# 5. User Statistics Summary

**What it does:** Counts unique contributors and lists all usernames for the filtered dataset

**Output:**
- Total unique user count
- Complete list of usernames who contributed images in the area
- Point count per user

**Use Case:** Understand contributor diversity, identify most active mappers in the region

In [ ]:
unique_users = gdf["username"].nunique()
user_list = sorted(gdf["username"].unique())

print(f"Year: {year}, Point count: {len(gdf)}, User count: {unique_users}")
print("Users:", ", ".join(user_list))

# 6. Multi-User Route Visualization

**What it does:** Creates a color-coded map showing routes/paths of multiple selected users

**Features:**
- **Color-Coded Routes**: Each user gets a unique color from seaborn palette
- **Selective Filtering**: Only shows pre-defined users of interest
- **OSM Background**: Buildings and roads provide geographic context
- **Legend**: User names with matching colors for easy identification

**Output:** 
- Interactive display
- PDF saved as `mapillary_users_routes.pdf`
- User contribution statistics printed

**Use Case:** Compare mapping patterns between users, identify frequently covered areas

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import osmnx as ox
from pathlib import Path
import seaborn as sns

# ------------------------------------
# 1. Path and user list
# ------------------------------------
MAP_DIR = Path("data") / "mapillary"
csv_path = MAP_DIR / "metadata.csv"

users_of_interest = [
    "chrisbeddow", "ira", "mapfool", "asturksever", "cihan10",
    "sessiznotalar", "lvl5", "selime", "oykukayaa",
    "jthitler", "vladimirr", "km2bp" ,"myusufuyan"
]

# ------------------------------------
# 2. Polygon definition
# ------------------------------------
polygon_coords = [
    (28.97201188246599, 41.027961514127306),
    (28.972796376763487, 41.02567058090895),
    (28.97595966022113, 41.02547966620972),
    (28.982589902348348, 41.028171512351044),
    (28.986841355315423, 41.02924058383375),
    (28.98772707468357, 41.02939330691432),
    (28.98954912595517, 41.02809514943808),
    (28.99051076412629, 41.02637696048352),
    (28.98522175418511, 41.02311227795534),
    (28.9803376445265, 41.02127940279426),
    (28.97727558613951, 41.02040113203624),
    (28.976187416630072, 41.02074480459719),
    (28.975200472191293, 41.02124121735267),
    (28.973783321202273, 41.02143214433928),
    (28.971784126057038, 41.02156579290059),
    (28.96986084971479, 41.02210038443376),
    (28.968443698725757, 41.02315046231212),
    (28.967684510695936, 41.02404778832296),
    (28.967279610413353, 41.02460145146428),
    (28.967380835484, 41.02536511712449),
    (28.968038798443192, 41.02584240366503),
    (28.96869676140238, 41.02668241957435),
    (28.96889921154367, 41.02694969511649),
    (28.97201188246599, 41.027961514127306)
]
polygon = Polygon(polygon_coords)

# ------------------------------------
# 3. Load data and create GeoDataFrame
# ------------------------------------
df = pd.read_csv(csv_path)
df["geometry"] = [Point(xy) for xy in zip(df.lon, df.lat)]
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

# Filter: user and inside polygon
gdf = gdf[gdf["username"].isin(users_of_interest)]
gdf = gdf[gdf.geometry.within(polygon)]

# ------------------------------------
# 4. Get OSM data (building + road)
# ------------------------------------
tags_bld = {"building": True}
tags_road = {"highway": True}

buildings = ox.features_from_polygon(polygon, tags_bld)
roads = ox.features_from_polygon(polygon, tags_road)

# ------------------------------------
# 5. Plot: colored routes by user
# ------------------------------------
fig, ax = plt.subplots(figsize=(16, 16))

# OSM buildings
if not buildings.empty:
    buildings.plot(ax=ax, facecolor="#eeeeee", edgecolor="gray", linewidth=0.3)

# OSM roads
if not roads.empty:
    roads.plot(ax=ax, color="lightgray", linewidth=0.6)

# User colors
palette = sns.color_palette("hls", len(users_of_interest))
user_color_map = dict(zip(users_of_interest, palette))

# Plot points by user
for user in users_of_interest:
    user_points = gdf[gdf["username"] == user]
    if not user_points.empty:
        user_points.plot(
            ax=ax,
            markersize=15,
            alpha=0.8,
            label=user,
            color=user_color_map[user]
        )

# Settings
ax.set_title("Mapillary Users Routes (Polygon + OSM)", fontsize=16)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_aspect("equal")
ax.legend(title="Users", loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

# Save as PDF
output_pdf = Path("mapillary_users_routes.pdf")
fig.savefig(output_pdf, format="pdf", dpi=300)


print("Selected Mapillary users with data in the polygon:")
for user in users_of_interest:
    user_data = gdf[gdf["username"] == user]
    count = len(user_data)
    print(f"- {user}: {count} points")

    active_users = [user for user in users_of_interest if not gdf[gdf["username"] == user].empty]
print(", ".join(active_users))

# 7. Organize Images by User into Separate Folders

**What it does:** Automatically organizes Karaköy images into user-specific subfolders

**Process:**
1. Reads all images from `data/mapillary/karakoy/` folder
2. Matches images with metadata to identify username
3. Creates a subfolder for each user
4. Copies each user's images to their respective folder

**Output:** Organized folder structure:
```
data/mapillary/karakoy/
├── mapfool/
├── chrisbeddow/
├── ira/
└── ...
```

**Use Case:** Prepare images for user-specific analysis or create individual user route animations

In [ ]:
import pandas as pd
from pathlib import Path
import shutil

# ------------------------------------
# 1. File paths
# ------------------------------------
MAP_DIR = Path("data") / "mapillary"
KARAKOY_DIR = MAP_DIR / "karakoy"
CSV_PATH = MAP_DIR / "metadata.csv"

df = pd.read_csv(CSV_PATH)

# ------------------------------------
# 2. Check image filenames
# ------------------------------------
image_files = list(KARAKOY_DIR.glob("*.jpg"))

# Match only with files that appear in metadata
df = df[df["filename"].isin([f.name for f in image_files])]

# ------------------------------------
# 3. Create folder for each user and copy files
# ------------------------------------
copied_count = 0

for _, row in df.iterrows():
    filename = row["filename"]
    username = row["username"]
    src = KARAKOY_DIR / filename
    user_dir = KARAKOY_DIR / username
    user_dir.mkdir(exist_ok=True)
    dst = user_dir / filename

    if src.exists():
        shutil.copy2(src, dst)
        copied_count += 1

print(f"{copied_count} images copied to respective user folders.")